# Vector Embeddings and RAG

Multi-document RAG pipeline over 4 AI policy documents. Compared MiniLM, MPNet, and multi-qa-MiniLM sentence transformers on retrieval quality and answer accuracy across 5 query types.

## Vector Embeddings and RAG

Vector Embeddings and RAG Starter Code
Multi-Document RAG System for AI Policy Analysis

In [2]:
# -*- coding: utf-8 -*-
# Install libraries and tools
# ----------------------------
!pip install -qU requests==2.32.4 chromadb langchain langchain-chroma langchain-huggingface langchain_openai langchain_community sentence-transformers tiktoken openai pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 124.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 999.8/999.8 kB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.2 MB/s et

In [3]:
# Imports
# ----------------------------
import os, re, shutil
import requests
import chromadb
from typing import List, Dict, Tuple
from IPython.display import Markdown, display

# LangChain imports
from langchain.schema import Document
from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains import LLMChain
from langchain_openai import ChatOpenAI, OpenAI
from langchain.prompts import PromptTemplate

## Part 1: Define Core Functions

In [4]:
# 1-1: Load and preprocess documents
# ----------------------------
def load_documents() -> List[Document]:
    """
    Load the four AI policy documents from their URLs.
    Returns a list of Document objects.
    """
    # (1) TODO: Implement this function
    # Load all four documents using appropriate LangChain document loaders
    # Handle both PDF and text documents
    # Return a list of Document objects

    url = "https://raw.githubusercontent.com/ntomuro/CSC583_2025Fall/main/HW3/data"
    document_filenames = [
        "Blueprint-for-an-AI-Bill-of-Rights.pdf", # The White House "Blueprint for an AI Bill of Rights" (2022)
        "2023-24283.pdf", # Executive Order on the Safe, Secure, and Trustworthy Development and Use of Artificial Intelligence (2023)
        "NIST.AI.100-1.pdf", # NIST AI Risk Management Framework (AI RMF 1.0) Overview
        "cellar_e0649735-a372-11eb-9585-01aa75ed71a1.0001.02_DOC_1.pdf" # The EU Artificial Intelligence Act (Key Principles Summary)
    ]
    ## Continue the TODO and collect raw documents in a variable 'documents'
    ## and return it.  You'll use langchain's PyPDFLoader, TextLoader to do.
    documents = []

    for filename in document_filenames:
        file_url = f"{url}/{filename}"
        response = requests.get(file_url)
        response.raise_for_status()
        with open(filename, "wb") as f: # saving file temporarily
            f.write(response.content)
        if filename.lower().endswith(".pdf"): # choosing appropriate loader
            loader = PyPDFLoader(filename)
        elif filename.lower().endswith(".txt"):
            loader = TextLoader(filename)
        else:
            continue  # skipping unsupported file types

        docs = loader.load()
        documents.extend(docs)
        os.remove(filename) # removing temp file

    return documents

def preprocess_documents(documents: List[Document]) -> List[Document]:
    """
    Split documents into chunks for vector storage.
    """
    # (2) TODO: Implement text splitting
    # Use RecursiveCharacterTextSplitter with appropriate (default) parameters.
    # Consider what chunk_size and chunk_overlap work best for multi-document retrieval
    # Return a list of document chunks from 'documents'.

    text_splitter = RecursiveCharacterTextSplitter( # using RecursiveCharacterTextSplitter with chunk_size=1000 and chunk_overlap=200
        chunk_size=1000, chunk_overlap=200)
    document_chunks = text_splitter.split_documents(documents) # splitting documents into smaller chunks


    # for now; change to something appropritate based on your code
    return document_chunks


# 1-2: Vector Store Population
# ----------------------------
def create_vector_store(documents: List[Document], embedding_model_name: str, client):
    """
    Create and return a retriever from documents using the specified embedding model.
    """
    # (3) TODO: Implement vector store creation
    # Initialize embedding function with the specified model.
    # Create Chroma vector store from documents.
    # Return a 'retriever' with search_kwargs={"k": 4}.

    embedding_function = HuggingFaceEmbeddings(model_name=embedding_model_name)
    vector_store = Chroma.from_documents(documents, embedding_function, client=client)
    retriever = vector_store.as_retriever(search_kwargs={"k": 4})



    return retriever

# 1-3: Query & Evaluation Set
# ----------------------------
def create_test_queries() -> List[str]:
    """
    Create a diverse set of 5 test queries as specified in the assignment.
    """
    # (4) TODO: Implement this function to return 5 well-designed test queries
    # Include:
    #   Single-Document Factual
    #   1 Multi-Document "Search & Combine"
    #   1 Thematic Synthesis
    #   1 Comparative
    #   1 Specific Definition/Scope
    # You can hard-code your own queries, or automatically generate queries in some way.

    queries = [
        "What does the AI Bill of Rights say about how automated systems should avoid discrimination?",# "Single-document factual question here",
        "How do the U.S. and EU define or handle high-risk AI — are they treating it the same way or differently?",# "Multi-document search and combine question here",
        "All these AI policies talk about transparency in some way — what are the main things they focus on?",# "Thematic synthesis question here",
        "How is accountability explained differently in the NIST framework versus the AI Bill of Rights?",# "Comparative question here",
        "What exactly counts as a high-risk AI system under the EU AI Act?"# "Specific definition/scope question here"
    ]
    return queries

# 1-4: The RAG Chain
# ----------------------------

def run_rag_query(query: str, retriever) -> Tuple[str, List[Document]]:
    """
    Execute a RAG query: retrieve relevant context and generate an answer.
    Returns the generated answer and the retrieved documents for evaluation.
    """
    # (5) TODO: Implement the RAG query execution
    # Retrieve relevant chunks using the retriever
    # Construct a high-quality prompt that includes context and query
    # Use an LLM chain ("gpt-3.5-turbo", with 'temperature=0') to generate the answer
    # Assign the answer and retrieved documents to variables 'answer' and
    # 'retrieved_docs', and return them.

    retrieved_docs = retriever.get_relevant_documents(query)
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs]) # formatting the retrieved context into a string
    prompt_template = PromptTemplate( # creating a clear prompt
        input_variables=["context", "question"],
        template="Answer the question based on the context below:\n\n{context}\n\nQuestion: {question}\nAnswer:")
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0) # using GPT-3.5 Turbo with temperature = 0 for deterministic results
    chain = LLMChain(llm=llm, prompt=prompt_template)
    answer = chain.run(context=context_text, question=query)







    return answer, retrieved_docs

# 1-5: Evaluation
# ----------------------------
def evaluate_results(query: str, retrieved_docs: List[Document], answer: str) -> Tuple[int, int]:
    """
    Manually evaluate the retrieval quality and answer quality.
    Returns (retrieval_score, answer_score) on scale 1-3.
    """
    # Manually evaluate and return scores
    # Retrieval Score: 1=Irrelevant, 2=Partially relevant, 3=Highly relevant
    # Answer Score: 1=Incorrect, 2=Partially correct, 3=Correct and comprehensive
    # Code is written.  You only enter your manual evaluation scores in real-time.
    print(f"Query: {query}")
    print(f"Retrieved {len(retrieved_docs)} documents")
    print(f"Answer: {answer}")
    print("---")

    retrieval_score = int(input("Retrieval score (1-3): "))
    answer_score = int(input("Answer score (1-3): "))

    return retrieval_score, answer_score

## Part 2: Experiment (main())


In [5]:
# 2: The Main Experiment
# ----------------------------
def main():
    """
    Main function to run the complete RAG experiment.
    """
    def safe_name(s):
        """
        Convert a string to a valid file name by removing invalid characters
        and using underscores to separate words.
        """
        return re.sub(r'[^0-9A-Za-z._-]', '_', s)

    # You can do in any way you like.
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.environ.get("HUGGINGFACE_HUB_TOKEN", "")

    # Define embedding models to test
    embedding_models = [
        "all-MiniLM-L6-v2",  # 384
        "all-mpnet-base-v2", # 768
        "multi-qa-MiniLM-L6-cos-v1" # 384
    ]

    # Load and preprocess documents
    print("Loading documents...")
    raw_documents = load_documents()
    print(f"Loaded {len(raw_documents)} raw documents")

    print("Preprocessing documents...")
    processed_documents = preprocess_documents(raw_documents)
    print(f"Created {len(processed_documents)} document chunks")

    # Create test queries
    test_queries = create_test_queries()
    print(f"Created {len(test_queries)} test queries")

    # Store results for analysis
    results = {}

    # Run experiment for each embedding model
    for model_name in embedding_models:
        print(f"\n{'='*50}")
        print(f"Testing model: {model_name}")
        print(f"{'='*50}")

        # Create a unique DB for the embedding model
        model_key = safe_name(model_name)
        db_path = f"./chroma_db_{model_key}"   # unique per model

        chroma_client = chromadb.PersistentClient(path=db_path)

        # Create vector store with current model
        retriever = create_vector_store(processed_documents, model_name, chroma_client)

        model_results = {
            'retrieval_scores': [],
            'answer_scores': [],
            'details': []
        }

        # Test each query
        for i, query in enumerate(test_queries): ####
            print(f"\nQuery {i+1}: {query}")

            # Run RAG query
            answer, retrieved_docs = run_rag_query(query, retriever)

            # Evaluate results
            retrieval_score, answer_score = evaluate_results(query, retrieved_docs, answer)

            # Store scores
            model_results['retrieval_scores'].append(retrieval_score)
            model_results['answer_scores'].append(answer_score)
            model_results['details'].append({
                'query': query,
                'answer': answer,
                'retrieved_docs': retrieved_docs
            })

        # Calculate averages
        model_results['avg_retrieval'] = sum(model_results['retrieval_scores']) / len(model_results['retrieval_scores'])
        model_results['avg_answer'] = sum(model_results['answer_scores']) / len(model_results['answer_scores'])

        results[model_name] = model_results

        print(f"\nModel {model_name} - Avg Retrieval: {model_results['avg_retrieval']:.2f}, Avg Answer: {model_results['avg_answer']:.2f}")

    return results, processed_documents, test_queries

## Part 3: Run Experiment

In [6]:
# Execute the experiment
final_results, documents, queries = main()
print (final_results)

Loading documents...
Loaded 265 raw documents
Preprocessing documents...
Created 1058 document chunks
Created 5 test queries

Testing model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Query 1: What does the AI Bill of Rights say about how automated systems should avoid discrimination?
Query: What does the AI Bill of Rights say about how automated systems should avoid discrimination?
Retrieved 4 documents
Answer: The AI Bill of Rights states that automated systems should not contribute to unjustified different treatment or impacts disfavoring people based on various protected classifications, as this would be considered algorithmic discrimination. Automated systems should be safe and effective, with protective measures in place to prevent discrimination and ensure the safety of individuals and communities.
---
Retrieval score (1-3): 3
Answer score (1-3): 3

Query 2: How do the U.S. and EU define or handle high-risk AI — are they treating it the same way or differently?
Query: How do the U.S. and EU define or handle high-risk AI — are they treating it the same way or differently?
Retrieved 4 documents
Answer: The EU defines and handles high-risk AI differently compar

<b> Print Results -- Run this cell after you finish writing your code.
Report the results table in your write-up!</b>

In [7]:
# Analysis Helper Functions
# ----------------------------

def print_results_table(results: Dict):
    """Print a formatted results table for the report."""
    print("\n" + "="*60)
    print("EXPERIMENT RESULTS SUMMARY")
    print("="*60)
    print(f"{'Model':<25} {'Avg Retrieval':<15} {'Avg Answer':<15}")
    print("-"*60)

    for model_name, model_results in results.items():
        print(f"{model_name:<25} {model_results['avg_retrieval']:<15.2f} {model_results['avg_answer']:<15.2f}")

def compare_retrieval_for_query(results: Dict, query_index: int):
    """Compare retrieval performance for a specific query across models."""
    print(f"\nRetrieval comparison for Query {query_index + 1}:")
    for model_name, model_results in results.items():
        retrieval_score = model_results['retrieval_scores'][query_index]
        answer_score = model_results['answer_scores'][query_index]
        print(f"  {model_name}: Retrieval={retrieval_score}, Answer={answer_score}")

# Print the final results
# ----------------------------
print_results_table(final_results)
for query in queries:
  compare_retrieval_for_query(final_results, queries.index(query))


EXPERIMENT RESULTS SUMMARY
Model                     Avg Retrieval   Avg Answer     
------------------------------------------------------------
all-MiniLM-L6-v2          3.00            2.40           
all-mpnet-base-v2         2.60            2.60           
multi-qa-MiniLM-L6-cos-v1 3.00            2.80           

Retrieval comparison for Query 1:
  all-MiniLM-L6-v2: Retrieval=3, Answer=3
  all-mpnet-base-v2: Retrieval=3, Answer=3
  multi-qa-MiniLM-L6-cos-v1: Retrieval=3, Answer=3

Retrieval comparison for Query 2:
  all-MiniLM-L6-v2: Retrieval=3, Answer=2
  all-mpnet-base-v2: Retrieval=2, Answer=2
  multi-qa-MiniLM-L6-cos-v1: Retrieval=3, Answer=2

Retrieval comparison for Query 3:
  all-MiniLM-L6-v2: Retrieval=3, Answer=3
  all-mpnet-base-v2: Retrieval=3, Answer=3
  multi-qa-MiniLM-L6-cos-v1: Retrieval=3, Answer=3

Retrieval comparison for Query 4:
  all-MiniLM-L6-v2: Retrieval=3, Answer=2
  all-mpnet-base-v2: Retrieval=2, Answer=2
  multi-qa-MiniLM-L6-cos-v1: Retrieval=3, Answ

### Testing different parameter values for all-MiniLM-L6-v2

In [9]:
# parameter combinations
param_combinations = [
    {"chunk_size": 500, "chunk_overlap": 50, "k": 4},
    {"chunk_size": 500, "chunk_overlap": 300, "k": 4},
    {"chunk_size": 1500, "chunk_overlap": 300, "k": 4},
    {"chunk_size": 500, "chunk_overlap": 100, "k": 2},
    {"chunk_size": 500, "chunk_overlap": 100, "k": 6},
]

print("Loading and preprocessing documents...")
raw_docs = load_documents()

for params in param_combinations:
    print("="*70)
    print(f"Running with chunk_size={params['chunk_size']}, overlap={params['chunk_overlap']}, k={params['k']}")
    print("="*70)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=params["chunk_size"],
        chunk_overlap=params["chunk_overlap"]
    )
    processed_docs = splitter.split_documents(raw_docs)

    chroma_client = chromadb.PersistentClient(path=f"./chroma_db_tuning_{params['chunk_size']}_{params['chunk_overlap']}_{params['k']}")
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_store = Chroma.from_documents(processed_docs, embeddings, client=chroma_client)
    retriever = vector_store.as_retriever(search_kwargs={"k": params["k"]})

    for query in create_test_queries():
        print(f"\nQuery: {query}")
        answer, retrieved_docs = run_rag_query(query, retriever)
        print(f"Retrieved {len(retrieved_docs)} documents")
        print(f"Answer: {answer}")
        print("---")

        # Manual evaluation
        retrieval_score = int(input("Retrieval score (1-3): "))
        answer_score = int(input("Answer score (1-3): "))
        print(f"Recorded → Retrieval={retrieval_score}, Answer={answer_score}")
        print("-"*60)

Loading and preprocessing documents...
Running with chunk_size=500, overlap=50, k=4

Query: What does the AI Bill of Rights say about how automated systems should avoid discrimination?
Retrieved 4 documents
Answer: The AI Bill of Rights states that automated systems should avoid algorithmic discrimination, which occurs when they contribute to unjustified different treatment or impacts disfavoring people based on various factors such as race, color, ethnicity, sex, religion, age, national origin, disability, veteran status, genetic information, or any other classification.
---
Retrieval score (1-3): 3
Answer score (1-3): 3
Recorded → Retrieval=3, Answer=3
------------------------------------------------------------

Query: How do the U.S. and EU define or handle high-risk AI — are they treating it the same way or differently?
Retrieved 4 documents
Answer: The EU defines and handles high-risk AI systems by establishing standards consistent with the Charter of fundamental rights of the Eu